# Tests de validation MSUC

Ce notebook lance les deux scripts de validation du dossier `examples/`, 
qui verifient les proprietes theoriques de l'article (encadrement borne/optimum/politique, 
equivalence des formulations 3-bin et intervalle - Prop. 3), puis propose une section 
d'exploration interactive (section 4) pour tester une instance, un nombre de scenarios 
et une methode au choix.

A executer depuis la racine du depot (le kernel Julia active automatiquement ce projet).

## 1. Installation des dependances (une seule fois)

Installe les paquets listes dans `Project.toml` (JuMP, Gurobi.jl, CSV, DataFrames, NCDatasets, ...). 
Necessite une licence Gurobi valide sur la machine.

In [1]:
import Pkg
Pkg.activate(@__DIR__)  # active ce projet, quel que soit le kernel Julia choisi
Pkg.instantiate()

  Activating project at `~/Documents/MSUC`


In [2]:
using MSUC, Gurobi, Printf, Random

GRB_ENV = Gurobi.Env()
optimizer = () -> Gurobi.Optimizer(GRB_ENV)

Set parameter Username
Set parameter LicenseID to value 2818369
Academic license - for non-commercial use only - expires 2027-05-05


#2 (generic function with 1 method)

In [3]:
INSTANCE    = "Data/T-Ramp/10_0_1_w.nc4"  # chemin vers le fichier .nc4
T           = 24                           # horizon (nombre de pas de temps)
S           = 5                            # nombre de scenarios (support de l'innovation ARMA)
FORCE       = 1.0                          # amplitude de l'alea (0.0 = deterministe)
ITERATIONS  = 30                           # nombre d'iterations SDDP
PRINT_LEVEL = 1                            # 0 = silencieux, 1 = log complet de MSUC.train

# :threebin  -> formulation 3-bin  (Sect. 2 de l'article, cuts (13))
# :extended  -> formulation intervalle (Sect. 3 de l'article, cuts (39))
METHOD = :extended

:extended

In [4]:
FORMULATIONS = Dict(
    :threebin => (MSUC.subproblem_builder_UC_def,     MSUC.SDDPFormulation,     "3-bin (13)"),
    :extended => (MSUC.subproblem_builder_UC_extended, MSUC.ExtendedFormulation, "intervalle (39)"),
)
builder, method, label = FORMULATIONS[METHOD]

instance = MSUC.parse_nc4(INSTANCE, optimizer, T; S = S)
@printf("instance %s : %d unites, %d intervalles, T = %d, S = %d, methode = %s\n",
        basename(INSTANCE), instance.N,
        sum(length(u.intervals) for u in instance.Thermalunits), T, S, label)

instance 10_0_1_w.nc4 : 10 unites, 2025 intervalles, T = 24, S = 5, methode = intervalle (39)


In [9]:
model = MSUC.PolicyGraph(
    builder, instance, FORCE, MSUC.Graph(T + 1), method;
    lower_bound = 0.0, optimizer = optimizer,
)

MSUC.train(model; iteration_limit = ITERATIONS, print_level = PRINT_LEVEL)

-------------------------------------------------------------------
         MSUC.jl (c) Oscar Dowson and contributors, 2017-25
-------------------------------------------------------------------
problem
  nodes           : 25
  state variables : 12
  scenarios       : 5.96046e+16
  existing cuts   : false
subproblem structure
  VariableRef                             : [79, 2780]
  AffExpr in MOI.EqualTo{Float64}         : [3, 997]
  AffExpr in MOI.GreaterThan{Float64}     : [1, 11]
  AffExpr in MOI.LessThan{Float64}        : [40, 960]
  VariableRef in MOI.EqualTo{Float64}     : [1, 1]
  VariableRef in MOI.GreaterThan{Float64} : [12, 2035]
  VariableRef in MOI.LessThan{Float64}    : [10, 10]
  VariableRef in MOI.Parameter{Float64}   : [50, 50]
  VariableRef in MOI.ZeroOne              : [730, 730]
numerical stability report
  matrix range     [1e-01, 1e+03]
  objective range  [1e+00, 3e+03]
  bounds range     [1e+02, 3e+02]
  rhs range        [1e+00, 1e+03]
---------------------------

### Resultats : trajectoire de la borne et cout de la politique simulee

In [6]:
log = model.most_recent_training_results.log
for l in log
    @printf("it %3d   borne = %.2f\n", l.iteration, l.bound)
end

it   1   borne = 58914.48
it   2   borne = 357938.91
it   3   borne = 380951.25
it   4   borne = 392122.99
it   5   borne = 400826.74
it   6   borne = 1803664.25
it   7   borne = 1805387.70
it   8   borne = 1809979.20
it   9   borne = 1815814.13
it  10   borne = 1815914.93
it  11   borne = 1816456.57
it  12   borne = 1817218.17
it  13   borne = 1818435.26
it  14   borne = 1819586.33
it  15   borne = 1821506.76
it  16   borne = 1821644.23
it  17   borne = 1821691.26
it  18   borne = 1822255.32
it  19   borne = 1822890.07
it  20   borne = 1822958.21
it  21   borne = 1823336.95
it  22   borne = 1823341.19
it  23   borne = 1824658.15
it  24   borne = 1824658.15
it  25   borne = 1824751.17
it  26   borne = 1824786.23
it  27   borne = 1824864.20
it  28   borne = 1824914.58
it  29   borne = 1825660.50
it  30   borne = 1826035.64


In [8]:
sim = MSUC.simulate(model)
@printf("\ncout de la politique simulee = %.2f  (%d chemins, exhaustif = %s)\n",
        sim.cost, sim.npaths, sim.exhaustive)
@printf("borne finale = %.2f  ->  encadrement relatif = %.2e\n",
        log[end].bound, (sim.cost - log[end].bound) / abs(log[end].bound))


cout de la politique simulee = 2007827.39  (1000 chemins, exhaustif = false)
borne finale = 1826035.64  ->  encadrement relatif = 9.96e-02
